In [2]:
from google.colab import files
uploaded = files.upload()  # upload your 5 saved files here
print("Uploaded:", list(uploaded.keys()))

Saving stress_audio_pipeline_3class.joblib to stress_audio_pipeline_3class.joblib
Saving stress_audio_pipeline_3class_meta.json to stress_audio_pipeline_3class_meta.json
Saving stress_text_pipeline.joblib to stress_text_pipeline.joblib
Saving stress_video_config.json to stress_video_config.json
Saving stress_video_resnet18.pt to stress_video_resnet18.pt
Uploaded: ['stress_audio_pipeline_3class.joblib', 'stress_audio_pipeline_3class_meta.json', 'stress_text_pipeline.joblib', 'stress_video_config.json', 'stress_video_resnet18.pt']


In [3]:
!pip -q install librosa opencv-python

import os, glob, json, random
import numpy as np
import joblib
import librosa

import torch
import torch.nn as nn
from PIL import Image
import torchvision.models as models
import torchvision.transforms as transforms

In [4]:
# If you have your ML text pipeline working well, we can swap later.
# For now, use the rule-based scorer that you verified works.

STRESS_WORDS = [
    "stress", "stressed", "overwhelmed", "anxious", "anxiety", "panic", "panicking",
    "can't sleep", "insomnia", "heart racing", "chest tight", "terrified", "fear",
    "hopeless", "breakdown", "can't cope", "helpless", "crying", "pressure"
]
CALM_WORDS = [
    "calm", "relaxed", "peaceful", "fine", "okay", "stable", "comfortable",
    "under control", "confident", "happy", "good", "great"
]

def text_score_rule(text: str) -> float:
    t = text.lower()
    stress_hits = sum(1 for w in STRESS_WORDS if w in t)
    calm_hits   = sum(1 for w in CALM_WORDS if w in t)
    raw = stress_hits - calm_hits
    score = 1 / (1 + (2.71828 ** (-1.2 * raw)))
    return float(score)

In [5]:
AUDIO_MODEL_PATH = "stress_audio_pipeline_3class.joblib"
audio_pipe = joblib.load(AUDIO_MODEL_PATH)
print("Audio classes:", audio_pipe.named_steps["clf"].classes_)

def extract_audio_features(path: str, sr: int = 16000) -> np.ndarray:
    y, sr = librosa.load(path, sr=sr)
    y, _ = librosa.effects.trim(y, top_db=25)

    target_len = int(sr * 1.0)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)))
    else:
        y = y[:target_len]

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std  = mfcc.std(axis=1)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr).mean(axis=1)

    zcr = float(librosa.feature.zero_crossing_rate(y).mean())
    rms = float(librosa.feature.rms(y=y).mean())

    return np.hstack([mfcc_mean, mfcc_std, chroma, [zcr, rms]]).astype(np.float32)

def audio_stress_score(audio_path: str) -> float:
    """Convert 3-class probs -> stress intensity in [0,1]."""
    x = extract_audio_features(audio_path).reshape(1, -1)
    proba = audio_pipe.predict_proba(x)[0]
    classes = list(audio_pipe.named_steps["clf"].classes_)

    def p(cls):
        return float(proba[classes.index(cls)]) if cls in classes else 0.0

    # Stress intensity: High=1, Moderate=0.5, Low=0
    return p("High") + 0.5 * p("Moderate")

Audio classes: ['High' 'Low' 'Moderate']


In [6]:
import cv2

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

IMG_SIZE = 224
img_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

VIDEO_MODEL_PATH = "stress_video_resnet18.pt"

video_model = models.resnet18(weights=None)
video_model.fc = nn.Linear(video_model.fc.in_features, 2)

state = torch.load(VIDEO_MODEL_PATH, map_location="cpu")
if isinstance(state, dict) and "state_dict" in state:
    state = state["state_dict"]

# rename keys (backbone./classifier. etc.)
fixed = {}
for k, v in state.items():
    if k.startswith("module."):
        k = k.replace("module.", "", 1)
    if k.startswith("backbone."):
        k = k.replace("backbone.", "", 1)
    if k.startswith("classifier."):
        k = k.replace("classifier.", "fc.", 1)
    fixed[k] = v

video_model.load_state_dict(fixed, strict=False)
video_model.to(device)
video_model.eval()

def sample_frames_cv2(video_path: str, num_frames: int = 16):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release()
        return None
    idx = np.linspace(0, total - 1, num_frames).astype(int).tolist()
    want = set(idx)
    frames = {}
    cur = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if cur in want:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames[cur] = frame
        cur += 1
    cap.release()
    out = [frames[i] for i in idx if i in frames]
    return out if len(out) > 0 else None

@torch.no_grad()
def video_stress_score(video_path: str, num_frames: int = 16) -> float:
    frames = sample_frames_cv2(video_path, num_frames=num_frames)
    if frames is None:
        return None
    batch = []
    for f in frames:
        batch.append(img_tf(Image.fromarray(f)))
    x = torch.stack(batch).to(device)
    logits = video_model(x)
    probs = torch.softmax(logits, dim=1)[:, 1]  # class1 prob
    return float(probs.mean().item())

Device: cpu


In [7]:
def level_from_score(p: float) -> str:
    if p < 0.33:
        return "Low"
    if p < 0.66:
        return "Moderate"
    return "High"

def predict_fused(text: str, audio_path: str, video_path: str,
                  w_text=0.35, w_audio=0.35, w_video=0.30):
    pt = text_score_rule(text)
    pa = audio_stress_score(audio_path)
    pv = video_stress_score(video_path)

    # normalize weights if any modality missing
    ws, ps = [], []
    if pt is not None: ws.append(w_text); ps.append(pt)
    if pa is not None: ws.append(w_audio); ps.append(pa)
    if pv is not None: ws.append(w_video); ps.append(pv)

    ws = np.array(ws, dtype=np.float32)
    ws = ws / ws.sum()

    p_final = float(np.sum(ws * np.array(ps, dtype=np.float32)))
    return {
        "p_text": float(pt),
        "p_audio": float(pa),
        "p_video": float(pv) if pv is not None else None,
        "p_final": p_final,
        "level": level_from_score(p_final)
    }

In [26]:
from google.colab import files
up = files.upload()  # upload 1 wav + 1 video
print(list(up.keys()))

Saving 01-01-03-02-01-01-01.avi to 01-01-03-02-01-01-01.avi
Saving 03-01-02-01-02-01-01.wav to 03-01-02-01-02-01-01.wav
['01-01-03-02-01-01-01.avi', '03-01-02-01-02-01-01.wav']


In [27]:
audio_path = [k for k in up.keys() if k.lower().endswith(".wav")][0]
video_path = [k for k in up.keys() if k.lower().endswith((".avi",".mp4",".mov",".mkv"))][0]

print(predict_fused(
    "I feel very calm and happy today.",
    audio_path,
    video_path
))

{'p_text': 0.0831728195975229, 'p_audio': 0.023240639291795585, 'p_video': 0.5222007036209106, 'p_final': 0.19390492141246796, 'level': 'Low'}


In [28]:
import json

fusion_config = {
    "fusion_weights": {
        "text": 0.35,
        "audio": 0.35,
        "video": 0.30
    },
    "classes": ["Low", "Moderate", "High"]
}

with open("stress_fusion_config.json", "w") as f:
    json.dump(fusion_config, f, indent=4)

print("Fusion config saved.")

Fusion config saved.


In [29]:
from google.colab import files
files.download("stress_fusion_config.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>